In [27]:
from utils.analysis.portfolio import PortfolioAnalyzer, PortfolioConfig
from utils.data import DataManager
from utils.tools.config import ANALYSIS_START_DATE, ANALYSIS_END_DATE

In [28]:
# PORTFOLIO CONFIGURATION
# Default values come from config.py (PORTFOLIO_CONFIG)
# Customize here if you need different values:

config = PortfolioConfig(
    min_score=60.0,              # Minimum quality score (default: 60.0)
    max_companies=10,            # Maximum companies in portfolio (default: 10)
    max_per_sector=3,            # Maximum per sector (default: 3)
    selection_method='quality',    # Method: 'balanced', 'value', 'growth', 'quality', 'total_score'
    weight_method='black_litterman',  # Method: 'equal', 'score', 'score_risk_adjusted', 'markowitz', 'black_litterman'
    lookback_years=3             # Historical years (default: 5)
)

print("[OK] Configuration created")
print(f"   Min Score: {config.min_score}")
print(f"   Max Companies: {config.max_companies}")
print(f"   Selection: {config.selection_method}")
print(f"   Weights: {config.weight_method}")
print(f"   Lookback: {config.lookback_years} years")

[OK] Configuration created
   Min Score: 60.0
   Max Companies: 10
   Selection: quality
   Weights: black_litterman
   Lookback: 3 years


In [29]:
# INITIALIZATION
# Create shared DataManager instance for efficiency
data_manager = DataManager()

# Create analyzer with configuration and data_manager
analyzer = PortfolioAnalyzer(config=config, data_manager=data_manager)

print("\nAnalysis dates (from config.py):")
print(f"   Start: {ANALYSIS_START_DATE}")
print(f"   End: {ANALYSIS_END_DATE}")
print("\n[OK] Analyzer initialized and ready to use")


Analysis dates (from config.py):
   Start: 2020-01-01
   End: 2025-12-24

[OK] Analyzer initialized and ready to use


In [30]:
# EXAMPLE 1: Analyze index components
# Available indices: 'SP500', 'NASDAQ100', 'DOW30', 'IBEX35', 'EUROSTOXX50', 'NIKKEI225', 'MSCI_WORLD'

print("Analyzing S&P 500...")
result = analyzer.analyze_from_index('MSCI_WORLD')

if result.get('success'):
    print(f"\n[OK] Portfolio created successfully!")
    print(f"\nSelected companies: {len(result['tickers'])}")
    for ticker in result['tickers']:
        weight = result['weights'].get(ticker, 0)
        print(f"   {ticker}: {weight*100:.2f}%")
    
    print(f"\nPortfolio metrics:")
    metrics = result['metrics']
    for key, value in metrics.items():
        print(f"   {key}: {value}")
    
    print(f"\nAnalyzed period:")
    print(f"   {result['period']['start']} -> {result['period']['end']}")
else:
    print(f"[ERROR] {result.get('error')}")

Analyzing S&P 500...
 MSCI World: combinando SP500 + EURO STOXX 50 + Nikkei 225
 Obtenidos 503 componentes del S&P 500
 Obtenidos 49 componentes del EURO STOXX 50
 Nikkei 225: usando lista fallback
 MSCI World (aprox): 777 empresas únicas
Period: 2020-01-01 → 2025-12-24

[OK] Portfolio created successfully!

Selected companies: 10
   4519.T: 10.44%
   DECK: 11.34%
   CME: 0.00%
   PLTR: 13.10%
   4507.T: 13.03%
   6532.T: 11.63%
   INCY: 6.97%
   APP: 12.69%
   6857.T: 11.34%
   MA: 9.46%

Portfolio metrics:
   total_score: 72.3199888658487
   sector_allocation: {'Healthcare': 0.30446223944942263, 'Consumer Cyclical': 0.1133898274345734, 'Financial Services': 0.09455910403598214, 'Technology': 0.24442168347378726, 'Industrials': 0.1162673255950993, 'Communication Services': 0.12689982001113537}
   num_companies: 10

Analyzed period:
   2020-01-01 -> 2025-12-24


In [31]:
# EXAMPLE 2: Analyze specific tickers
# Useful for custom portfolios

custom_tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'TSLA', 'JPM', 'V', 'WMT']

print(f"Analyzing {len(custom_tickers)} custom companies...")
result = analyzer.analyze(custom_tickers)

if result.get('success'):
    print(f"\n[OK] Portfolio created!")
    print(f"Selected: {len(result['tickers'])}/{len(custom_tickers)}")
    print(f"\nWeight distribution:")
    for ticker in result['tickers']:
        weight = result['weights'].get(ticker, 0)
        score = result['analysis'][ticker]['scores']['total']
        print(f"   {ticker}: {weight*100:5.2f}% (Score: {score:.1f})")
else:
    print(f"[ERROR] {result.get('error')}")

Analyzing 10 custom companies...
Period: 2020-01-01 → 2025-12-24

[OK] Portfolio created!
Selected: 3/10

Weight distribution:
   NVDA: 42.18% (Score: 68.6)
   V: 30.59% (Score: 70.4)
   MSFT: 27.23% (Score: 62.7)


In [32]:
# EXAMPLE 3: Compare different selection methods
# Available methods: 'balanced', 'value', 'growth', 'total_score'

test_tickers = ['AAPL', 'MSFT', 'GOOGL', 'META', 'NVDA', 'JPM', 'BAC', 'WMT', 'PG', 'JNJ']
methods = ['balanced', 'value', 'growth', 'quality', 'total_score']

print("Comparing selection methods:\n")

for method in methods:
    config_test = PortfolioConfig(
        min_score=60.0,
        max_companies=5,
        selection_method=method,
        weight_method='equal'
    )
    
    analyzer_test = PortfolioAnalyzer(config=config_test, data_manager=data_manager)
    result = analyzer_test.analyze(test_tickers)
    
    if result.get('success'):
        print(f"Method '{method}':")
        print(f"   Selected: {', '.join(result['tickers'])}")
    else:
        print(f"[ERROR] Method '{method}': {result.get('error')}")
    print()

Comparing selection methods:

Method 'balanced':
   Selected: NVDA, MSFT

Method 'value':
   Selected: MSFT, NVDA

Method 'growth':
   Selected: NVDA, MSFT

Method 'quality':
   Selected: NVDA, MSFT

Method 'total_score':
   Selected: NVDA, MSFT



In [33]:
# EXAMPLE 4: Compare weight optimization methods
# Methods: 'equal', 'score', 'score_risk_adjusted', 'markowitz', 'black_litterman'

weight_methods = {
    'equal': 'Equal Weight (1/n)',
    'score': 'Score-Based (proportional to fundamentals)',
    'score_risk_adjusted': 'Score/Risk (score / volatility)',
    'markowitz': 'Markowitz (max Sharpe Ratio)',
    'black_litterman': 'Black-Litterman (equilibrium + views)',
}

test_tickers = ['AAPL', 'MSFT', 'GOOGL', 'META', 'NVDA', 'JPM', 'BAC', 'WMT', 'PG', 'JNJ']

print("Comparing weight optimization methods:\n")

for method_key, method_name in weight_methods.items():
    config_test = PortfolioConfig(
        min_score=60.0,
        max_companies=8,
        selection_method='balanced',
        weight_method=method_key,
        lookback_years=3
    )
    
    analyzer_test = PortfolioAnalyzer(config=config_test, data_manager=data_manager)
    result = analyzer_test.analyze(test_tickers)
    
    if result.get('success'):
        print(f"{method_name}:")
        # Sort by weight descending
        sorted_weights = sorted(result['weights'].items(), key=lambda x: x[1], reverse=True)
        for ticker, weight in sorted_weights:
            if weight > 0.001:  # Only show weights > 0.1%
                print(f"   {ticker}: {weight*100:5.2f}%")
        print(f"   Weighted score: {result['metrics']['total_score']:.1f}")
    else:
        print(f"[ERROR] {method_name}: {result.get('error')}")
    print()

Comparing weight optimization methods:

Equal Weight (1/n):
   NVDA: 50.00%
   MSFT: 50.00%
   Weighted score: 65.7

Score-Based (proportional to fundamentals):
   NVDA: 52.24%
   MSFT: 47.76%
   Weighted score: 65.8

Period: 2020-01-01 → 2025-12-24
Score/Risk (score / volatility):
   MSFT: 62.20%
   NVDA: 37.80%
   Weighted score: 64.9

Period: 2020-01-01 → 2025-12-24
Markowitz (max Sharpe Ratio):
   NVDA: 100.00%
   Weighted score: 68.6

Period: 2020-01-01 → 2025-12-24
Black-Litterman (equilibrium + views):
   NVDA: 61.35%
   MSFT: 38.65%
   Weighted score: 66.3



In [34]:
# EXAMPLE 5: Professional portfolio with Black-Litterman
# BL combines market equilibrium + your fundamental scores
# It is the most widely used model in institutional asset management

print("Creating professional Black-Litterman portfolio...\n")

config_bl = PortfolioConfig(
    min_score=60.0,
    max_companies=10,
    max_per_sector=3,
    selection_method='quality',
    weight_method='black_litterman',
    lookback_years=5
)

analyzer_bl = PortfolioAnalyzer(config=config_bl, data_manager=data_manager)
result = analyzer_bl.analyze_from_index('MSCI_WORLD')

if result.get('success'):
    print("[OK] Black-Litterman portfolio created!\n")
    
    print("Optimized weights:")
    sorted_weights = sorted(result['weights'].items(), key=lambda x: x[1], reverse=True)
    for ticker, weight in sorted_weights:
        if weight > 0.001:
            score = result['analysis'][ticker]['scores']['total']
            print(f"   {ticker}: {weight*100:5.2f}%  (Score: {score:.1f})")
    
    print(f"\nMetrics:")
    print(f"   Weighted score: {result['metrics']['total_score']:.1f}")
    print(f"   Companies: {result['metrics']['num_companies']}")
    print(f"\nSector diversification:")
    for sector, alloc in sorted(result['metrics']['sector_allocation'].items(), key=lambda x: x[1], reverse=True):
        print(f"   {sector}: {alloc*100:.1f}%")
    
    print(f"\nPeriod: {result['period']['start']} -> {result['period']['end']}")
else:
    print(f"[ERROR] {result.get('error')}")

Creating professional Black-Litterman portfolio...

 MSCI World: combinando SP500 + EURO STOXX 50 + Nikkei 225
 Obtenidos 503 componentes del S&P 500
 Obtenidos 49 componentes del EURO STOXX 50
 Nikkei 225: usando lista fallback
 MSCI World (aprox): 777 empresas únicas
Period: 2020-01-01 → 2025-12-24
[OK] Black-Litterman portfolio created!

Optimized weights:
   PLTR: 13.10%  (Score: 65.4)
   4507.T: 13.03%  (Score: 82.3)
   APP: 12.69%  (Score: 68.1)
   6532.T: 11.63%  (Score: 73.7)
   6857.T: 11.34%  (Score: 71.0)
   DECK: 11.34%  (Score: 71.8)
   4519.T: 10.44%  (Score: 71.9)
   MA:  9.46%  (Score: 73.9)
   INCY:  6.97%  (Score: 73.6)

Metrics:
   Weighted score: 72.3
   Companies: 10

Sector diversification:
   Healthcare: 30.4%
   Technology: 24.4%
   Communication Services: 12.7%
   Industrials: 11.6%
   Consumer Cyclical: 11.3%
   Financial Services: 9.5%

Period: 2020-01-01 -> 2025-12-24
